In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import t
from scipy.stats import ttest_rel

In [2]:
root_path = Path.cwd().parent
corpus_name = "20260702_configurations_comparison_corpus"
evaluation_results_path = Path(os.path.join(root_path, "evaluations"))

In [3]:
def compare_collections(corpus_name, collection_names, configuration_name, ref_collection_name, other_collection_names, metric):
    mrr_results = pd.DataFrame()

    for collection_name in collection_names:
        configuration_path = os.path.join(
            evaluation_results_path,
            corpus_name,
            collection_name,
            configuration_name,
            "df_scores_ordered.csv"
        )
        config_results_df = pd.read_csv(configuration_path)
        mrr_results[collection_name] = config_results_df[metric].values

    reference = ref_collection_name
    others = other_collection_names
    ttest_results_list = []
    for other in others:
        ref_values = mrr_results[reference]
        other_values = mrr_results[other]
        diff = ref_values - other_values
        t_stat, p_value = ttest_rel(ref_values, other_values)
        n = len(diff)
        dfree = n - 1
        mean_diff = diff.mean()
        std_diff = diff.std(ddof=1)
        se_diff = std_diff / np.sqrt(n)
        ci_low, ci_high = t.interval(
            confidence=0.95,
            df=dfree,
            loc=mean_diff,
            scale=se_diff
        )
        ttest_results_list.append({
            "metric": metric,
            "comparison": f"{reference} vs {other}",
            "n": n,
            "mean_ref": ref_values.mean(),
            "mean_other": other_values.mean(),
            "mean_diff": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "t": t_stat,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

    ttest_results_df = pd.DataFrame(ttest_results_list)

    return ttest_results_df

In [4]:
def compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric):
    ref_name = configuration_names[0]
    other_name = configuration_names[1]

    ttest_results_list = []

    for collection_name in collection_names:
        ref_path = os.path.join(
            evaluation_results_path    ,
            corpus_name,
            collection_name,
            ref_name,
            "df_scores_ordered.csv"
        )

        other_path = os.path.join(
            evaluation_results_path    ,
            corpus_name,
            collection_name,
            other_name,
            "df_scores_ordered.csv"
        )

        ref_df = pd.read_csv(ref_path)
        other_df = pd.read_csv(other_path)

        ref_values = ref_df[metric].values
        other_values = other_df[metric].values

        diff = ref_values - other_values
        t_stat, p_value = ttest_rel(ref_values, other_values)
        n = len(diff)
        dfree = n - 1
        mean_diff = diff.mean()
        std_diff = diff.std(ddof=1)
        se_diff = std_diff / np.sqrt(n)
        ci_low, ci_high = t.interval(
            confidence=0.95,
            df=dfree,
            loc=mean_diff,
            scale=se_diff
        )
        ttest_results_list.append({
            "metric": metric,
            "collection": collection_name,
            "comparison": f"{ref_name} vs {other_name}",
            "n": n,
            "mean_ref": ref_values.mean(),
            "mean_other": other_values.mean(),
            "mean_diff": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "t": t_stat,
            "df": dfree,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

    ttest_results_df = pd.DataFrame(ttest_results_list)

    return ttest_results_df

In [5]:
def compare_one_configuration_vs_others_for_collection(corpus_name, collection_name, ref_configuration_name, other_configuration_names, metric):

    ref_path = os.path.join(
        evaluation_results_path    ,
        corpus_name,
        collection_name,
        ref_configuration_name,
        "df_scores_ordered.csv"
    )

    ref_df = pd.read_csv(ref_path)
    ref_values = ref_df[metric].values

    ttest_results_list = []

    for other in other_configuration_names:
        other_path = os.path.join(
            evaluation_results_path    ,
            corpus_name,
            collection_name,
            other,
            "df_scores_ordered.csv"
        )

        other_df = pd.read_csv(other_path)
        other_values = other_df[metric].values

        diff = ref_values - other_values
        t_stat, p_value = ttest_rel(ref_values, other_values)
        n = len(diff)
        dfree = n - 1
        mean_diff = diff.mean()
        std_diff = diff.std(ddof=1)
        se_diff = std_diff / np.sqrt(n)
        ci_low, ci_high = t.interval(
            confidence=0.95,
            df=dfree,
            loc=mean_diff,
            scale=se_diff
        )
        ttest_results_list.append({
            "metric": metric,
            "collection": collection_name,
            "comparison": f"{ref_configuration_name} vs {other}",
            "n": n,
            "mean_ref": ref_values.mean(),
            "mean_other": other_values.mean(),
            "mean_diff": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "t": t_stat,
            "df": dfree,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

    ttest_results_df = pd.DataFrame(ttest_results_list)

    return ttest_results_df

# Comparaisons retrieval

## Ttest pour savoir si configuration_e significativement meilleure sur dense + no reranker par rapport aux autres collections -> pas le cas

```
PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_a" --run-name="configuration_a_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:13:58.585693Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_c" --run-name="configuration_c_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:25.599367Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_d" --run-name="configuration_d_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:32.555443Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_f" --run-name="configuration_f_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:44.643530Z" --name-dir="retrieval_dense_no_reranker"
```

In [6]:
collection_names = ["configuration_a", "configuration_b", "configuration_c", "configuration_d", "configuration_e", "configuration_f"]
configuration_name = "retrieval_dense_no_reranker"
ref_collection_name = "configuration_e"
other_collection_names = ["configuration_a", "configuration_b", "configuration_c", "configuration_d", "configuration_f"]
metric = "mrr_doc"

retrieval_ttest_results_dense_no_reranked_df = compare_collections(corpus_name, collection_names, configuration_name, ref_collection_name, other_collection_names, metric)

retrieval_ttest_results_dense_no_reranked_df

,metric,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,p_value,significant
0,mrr_doc,configuration_e vs configuration_a,13,0.026426,0.000000,0.026426,-0.009306,0.062158,1.611386,0.133069,False
1,mrr_doc,configuration_e vs configuration_b,13,0.026426,0.014685,0.011741,-0.029900,0.053382,0.614327,0.550469,False
2,mrr_doc,configuration_e vs configuration_c,13,0.026426,0.000000,0.026426,-0.009306,0.062158,1.611386,0.133069,False
3,mrr_doc,configuration_e vs configuration_d,13,0.026426,0.013889,0.012537,-0.032853,0.057927,0.601817,0.558495,False
4,mrr_doc,configuration_e vs configuration_f,13,0.026426,0.015385,0.011042,-0.041132,0.063215,0.461110,0.652965,False


## Ttest pour savoir si dense VS hybrid pour collection_b et collection_e est significatif -> c'est le cas

In [7]:
collection_names = ["configuration_b", "configuration_e"]
configuration_names = ["retrieval_dense_no_reranker", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_dense_vs_hybrid_no_reranked_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_dense_vs_hybrid_no_reranked_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_dense_no_reranker vs retrieval_hybri...,13,0.014685,0.384615,-0.369930,-0.605484,-0.134376,-3.421760,12,0.005062,True
1,mrr_doc,configuration_e,retrieval_dense_no_reranker vs retrieval_hybri...,13,0.026426,0.356410,-0.329984,-0.549645,-0.110323,-3.273108,12,0.006665,True


## Ttest pour savoir si sparse VS hybrid pour collection_b et collection_e est significatif -> pas le cas (mais sparse quand meme meilleur que hybrid selon certains recall@k observes)

In [8]:
collection_names = ["configuration_b", "configuration_e"]
configuration_names = ["retrieval_sparse_no_reranker", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_sparse_vs_hybrid_no_reranked_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_sparse_vs_hybrid_no_reranked_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_sparse_no_reranker vs retrieval_hybr...,13,0.546154,0.384615,0.161538,-0.173821,0.496898,1.049508,12,0.314622,False
1,mrr_doc,configuration_e,retrieval_sparse_no_reranker vs retrieval_hybr...,13,0.546154,0.356410,0.189744,-0.178306,0.557793,1.123261,12,0.283297,False


## Ttest pour savoir si l'un des rerankers est significativement meilleur pour collection_b et collection_e -> pas le cas (sur dense et hybrid, sparse on sait pas)

In [9]:
collection_names = ["configuration_b", "configuration_e"]
configuration_names = ["retrieval_dense_reranker_bge", "retrieval_dense_reranker_qwen"]
metric = "mrr_doc"

retrieval_ttest_results_dense_rerankers_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_dense_rerankers_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.097436,0.104701,-0.007265,-0.261177,0.246647,-0.062341,12,0.951318,False
1,mrr_doc,configuration_e,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.086538,0.042308,0.044231,-0.140194,0.228655,0.522547,12,0.610801,False


In [10]:
collection_names = ["configuration_b", "configuration_e"]
configuration_names = ["retrieval_hybrid_reranker_bge", "retrieval_hybrid_reranker_qwen"]
metric = "mrr_doc"

retrieval_ttest_results_hybrid_rerankers_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_hybrid_rerankers_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.635294,0.560562,0.074732,-0.230528,0.379993,0.533407,12,0.603491,False
1,mrr_doc,configuration_e,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.641758,0.511538,0.130220,-0.232340,0.492780,0.782559,12,0.449054,False


## Ttest pour savoir si le reranker bge ameliore significativement sparse, dense et hybrid pour collection_e -> malheureusement pas le cas

In [11]:
collection_names = ["configuration_e"]
configuration_names = ["retrieval_sparse_reranker_bge", "retrieval_sparse_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_sparses_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_sparses_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_sparse_reranker_bge vs retrieval_spa...,13,0.700855,0.546154,0.154701,-0.211222,0.520624,0.921135,12,0.375128,False


In [12]:
collection_names = ["configuration_e"]
configuration_names = ["retrieval_dense_reranker_bge", "retrieval_dense_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_denses_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_denses_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.086538,0.026426,0.060112,-0.114901,0.235125,0.748364,12,0.468655,False


In [13]:
collection_names = ["configuration_e"]
configuration_names = ["retrieval_hybrid_reranker_bge", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_hybrids_df = compare_paired_configurations_per_collections(corpus_name, collection_names, configuration_names, metric)

retrieval_ttest_results_hybrids_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.641758,0.35641,0.285348,-0.116503,0.687199,1.54714,12,0.147786,False


## Configuration finale pour les hyperparametres du retrieval

- taille des chunks de 2'000 caracteres
- qwen comme modele d'embeddings
- mode de recuperation hybride
- bge comme reranker

# Comparaisons generation

In [14]:
def get_ttest_generation_results(ref_configuration_name):
    corpus_name = "20260706_last_corpus"
    collection_name = "20260706_last_collection"
    all_configuration_names = [
        "generation_mistral_2_chunks_mistral_judge",
        "generation_mistral_5_chunks_mistral_judge",
        "generation_mistral_20_chunks_mistral_judge",
        "generation_mistral_max_10_chunks_mistral_judge",
        "generation_mistral_2_docs_mistral_judge",
        "generation_mistral_modular_context_mistral_judge",
        "generation_mistral_modular_context_item_referenced_mistral_judge"
    ]
    other_configuration_names = [all_configuration_name for all_configuration_name in all_configuration_names if all_configuration_name != ref_configuration_name]

    metrics = ["semantic_similarity", "Groundedness (Faithfulness-RAGAS)", "Answer Relevance (Relevance-Langfuse)"]

    print(f"Comparisons for {ref_configuration_name}\n")
    for metric in metrics:
        print(f"    Results for {metric}:")
        generation_ttest_results_df = compare_one_configuration_vs_others_for_collection(corpus_name,
                                                                                     collection_name,
                                                                                     ref_configuration_name,
                                                                                     other_configuration_names,
                                                                                     metric)
        display(generation_ttest_results_df)

## Ttest pour savoir si la maniere la plus reflechie pour construire le contexte est significativement meilleure -> malheureusement pas le cas

In [15]:
get_ttest_generation_results("generation_mistral_modular_context_item_referenced_mistral_judge")

Comparisons for generation_mistral_modular_context_item_referenced_mistral_judge

    Results for semantic_similarity:


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.573995,0.003230,-0.160619,0.167078,0.042947,12,0.966450,False
1,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.542732,0.034493,-0.131418,0.200405,0.452981,12,0.658644,False
2,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.605182,-0.027957,-0.180410,0.124495,-0.399557,12,0.696503,False
3,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.525505,0.051720,-0.133866,0.237307,0.607205,12,0.555031,False
4,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.607213,-0.029988,-0.194313,0.134337,-0.397615,12,0.697896,False
5,semantic_similarity,20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.577225,0.532453,0.044772,-0.076391,0.165935,0.805118,12,0.436414,False


    Results for Groundedness (Faithfulness-RAGAS):


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.871538,-0.280462,-0.639539,0.078616,-1.701785,12,0.114537,False
1,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.865385,-0.274308,-0.608587,0.059972,-1.787920,12,0.099048,False
2,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.843592,-0.252515,-0.573520,0.068489,-1.713945,12,0.112227,False
3,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.788462,-0.197385,-0.582299,0.187530,-1.117297,12,0.285738,False
4,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.792308,-0.201231,-0.512519,0.110058,-1.408482,12,0.184367,False
5,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.591077,0.769231,-0.178154,-0.483711,0.127403,-1.270350,12,0.228038,False


    Results for Answer Relevance (Relevance-Langfuse):


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.930769,-0.007692,-0.231718,0.216334,-0.074813,12,0.941596,False
1,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.846154,0.076923,-0.153389,0.307235,0.727714,12,0.480747,False
2,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.923077,0.000000,-0.210782,0.210782,0.000000,12,1.000000,False
3,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.803846,0.119231,-0.018977,0.257438,1.879651,12,0.084648,False
4,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.876923,0.046154,-0.200073,0.292381,0.408406,12,0.690169,False
5,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_modular_context_item_refere...,13,0.923077,0.838462,0.084615,-0.082426,0.251657,1.103685,12,0.291371,False


## Ttest pour savoir si envoyer deux documents au contexte est significativement meilleur -> pas le cas non plus

In [16]:
get_ttest_generation_results("generation_mistral_2_docs_mistral_judge")

Comparisons for generation_mistral_2_docs_mistral_judge

    Results for semantic_similarity:


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.573995,0.033218,-0.117824,0.184260,0.479171,12,0.640431,False
1,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.542732,0.064481,-0.095066,0.224029,0.880571,12,0.395844,False
2,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.605182,0.002031,-0.168507,0.172569,0.025945,12,0.979727,False
3,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.525505,0.081708,-0.094597,0.258014,1.009766,12,0.332535,False
4,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.532453,0.074760,-0.100827,0.250348,0.927677,12,0.371859,False
5,semantic_similarity,20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.607213,0.577225,0.029988,-0.134337,0.194313,0.397615,12,0.697896,False


    Results for Groundedness (Faithfulness-RAGAS):


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.871538,-0.079231,-0.363597,0.205135,-0.607067,12,0.555119,False
1,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.865385,-0.073077,-0.400971,0.254818,-0.485586,12,0.636006,False
2,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.843592,-0.051285,-0.297381,0.194812,-0.454048,12,0.657897,False
3,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.788462,0.003846,-0.297156,0.304848,0.027841,12,0.978247,False
4,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.769231,0.023077,-0.362405,0.408559,0.130435,12,0.898384,False
5,Groundedness (Faithfulness-RAGAS),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.792308,0.591077,0.201231,-0.110058,0.512519,1.408482,12,0.184367,False


    Results for Answer Relevance (Relevance-Langfuse):


,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.930769,-0.053846,-0.194737,0.087045,-0.832704,12,0.421276,False
1,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.846154,0.030769,-0.212685,0.274224,0.275371,12,0.787715,False
2,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.923077,-0.046154,-0.270389,0.178081,-0.448461,12,0.661811,False
3,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.803846,0.073077,-0.181103,0.327256,0.626411,12,0.542778,False
4,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.838462,0.038462,-0.256075,0.332998,0.284517,12,0.780865,False
5,Answer Relevance (Relevance-Langfuse),20260706_last_collection,generation_mistral_2_docs_mistral_judge vs gen...,13,0.876923,0.923077,-0.046154,-0.292381,0.200073,-0.408406,12,0.690169,False


## Configuration finale pour les hyperparametres de generation

- prompts:
    - PROMPT_TEMPLATE_FR="Réponds à la question en utilisant uniquement le contexte fourni.\n\nContexte:\n{context_text}\n\nQuestion:\n{query}\n\nRéponse:"
    - PROMPT_TEMPLATE_EN="Answer the question using only the provided context.\n\nContext:\n{context_text}\n\nQuestion:\n{query}\n\nAnswer:"
- mistral comme LLM
- contexte contenant soit des documents, des chunks ou un mix des deux selon apparitions repetees de chunks venant d'un meme document et score superieur a un seuil de 0.2 pour chunks uniques
- mistral comme LLM-as-Judge (et qwen pour les embeddings)